# Phase 4: Full Feature Matrix & Training Labels

**Goal:** Merge demographics and provider features into one unified matrix, construct binary training labels using 12-month comparison windows, and prepare data for model training.

**Inputs:**
- `cb_demographics.parquet` (Phase 2)
- `cb_provider_features.parquet` (Phase 3)

**Outputs:**
- `feature_matrix_full.parquet`
- `training_data.parquet`
- `feature_columns.json`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
DEMOGRAPHICS_FILE = '/Users/yugeshchandraroy/Downloads/Projects/market-demographics-fcc-pipeline/output/cb_demographics.parquet'
PROVIDER_FEATURES_FILE = '/Users/yugeshchandraroy/Downloads/Projects/broadband-market-scanner/data/intermediate/cb_provider_features.parquet'
OUTPUT_DIR = '/Users/yugeshchandraroy/Downloads/Projects/broadband-market-scanner/data/intermediate'

print('Phase 4: Full Feature Matrix & Training Labels')
print('=' * 55)

## Step 1: Load Inputs

In [ ]:
# Load demographics (period-independent, one row per CB)
demographics = pd.read_parquet(DEMOGRAPHICS_FILE)
print(f'Demographics: {demographics.shape}')
print(f'  Columns: {list(demographics.columns)}')
print(f'  CB count: {demographics["cb_fips"].nunique():,}')

print()

# Load provider features (one row per CB per filing period)
provider = pd.read_parquet(PROVIDER_FEATURES_FILE)
print(f'Provider features: {provider.shape}')
print(f'  Columns: {list(provider.columns)}')
print(f'  Filing periods: {sorted(provider["filing_period"].unique())}')
print(f'  CB count: {provider["block_geoid"].nunique():,}')

## Step 2: Merge Into Full Feature Matrix

In [ ]:
# Merge demographics (period-independent) with provider features (per period)
# Join key: cb_fips (demographics) == block_geoid (provider features)

feature_matrix = provider.merge(
    demographics,
    left_on='block_geoid',
    right_on='cb_fips',
    how='left'
)

# Drop duplicate cbg_fips column (exists in both tables)
if 'cbg_fips_x' in feature_matrix.columns:
    feature_matrix.rename(columns={'cbg_fips_x': 'cbg_fips'}, inplace=True)
    feature_matrix.drop(columns=['cbg_fips_y'], inplace=True, errors='ignore')

# Drop duplicate cb_fips (we keep block_geoid as the canonical ID)
feature_matrix.drop(columns=['cb_fips'], inplace=True, errors='ignore')

print(f'Feature matrix: {feature_matrix.shape}')
print(f'  Rows per period:')
for period in sorted(feature_matrix['filing_period'].unique()):
    count = len(feature_matrix[feature_matrix['filing_period'] == period])
    print(f'    {period}: {count:,}')

# Check merge quality
demo_matched = feature_matrix['total_housing_units'].notna().sum()
demo_missing = feature_matrix['total_housing_units'].isna().sum()
print(f'\nDemographics match: {demo_matched:,} matched, {demo_missing:,} missing ({100*demo_missing/len(feature_matrix):.2f}%)')

In [ ]:
# Summary statistics of the merged matrix
print('Feature Matrix Summary')
print('=' * 50)
print(f'Total rows: {len(feature_matrix):,}')
print(f'Total columns: {len(feature_matrix.columns)}')
print(f'Unique blocks: {feature_matrix["block_geoid"].nunique():,}')
print(f'Filing periods: {sorted(feature_matrix["filing_period"].unique())}')
print(f'\nColumn list:')
for i, col in enumerate(feature_matrix.columns):
    null_pct = 100 * feature_matrix[col].isna().sum() / len(feature_matrix)
    print(f'  {i+1:2d}. {col:40s} nulls: {null_pct:.1f}%')

## Step 3: Construct Training Labels (12-Month Comparison Windows)

The prediction target: **did this CB gain fiber over a 12-month period?**

Comparison pairs (June-to-June):
- Features from Jun '22 → Label: gained fiber by Jun '23?
- Features from Jun '23 → Label: gained fiber by Jun '24?

In [ ]:
# Define 12-month comparison pairs
periods = sorted(feature_matrix['filing_period'].unique())
print(f'Available periods: {periods}')

# Build pairs: June-to-June (12 months apart)
comparison_pairs = []
for i, p1 in enumerate(periods):
    month1 = p1.split('-')[1]  # '06' or '12'
    for p2 in periods[i+1:]:
        month2 = p2.split('-')[1]
        year1 = int(p1.split('-')[0])
        year2 = int(p2.split('-')[0])
        # 12-month gap: same month, consecutive years
        if month1 == month2 and year2 - year1 == 1:
            comparison_pairs.append((p1, p2))

print(f'\n12-month comparison pairs:')
for t, t12 in comparison_pairs:
    print(f'  Features from {t} → Label: gained fiber by {t12}?')

In [ ]:
# Construct training labels
training_sets = []

for train_period, label_period in comparison_pairs:
    print(f'\nBuilding training set: {train_period} → {label_period}')
    
    # Features from time t
    features_t = feature_matrix[feature_matrix['filing_period'] == train_period].copy()
    
    # Fiber status at time t+12mo
    fiber_t12 = feature_matrix[feature_matrix['filing_period'] == label_period][['block_geoid', 'has_fiber']].copy()
    fiber_t12.rename(columns={'has_fiber': 'has_fiber_future'}, inplace=True)
    
    # Filter to CBs that did NOT have fiber at time t
    no_fiber_t = features_t[features_t['has_fiber'] == 0].copy()
    print(f'  CBs without fiber at {train_period}: {len(no_fiber_t):,}')
    
    # Merge with future fiber status
    labeled = no_fiber_t.merge(fiber_t12, on='block_geoid', how='inner')
    print(f'  CBs matched to {label_period}: {len(labeled):,}')
    
    # Label: gained_fiber = 1 if went from 0 to 1
    labeled['gained_fiber'] = labeled['has_fiber_future'].astype(int)
    labeled.drop(columns=['has_fiber_future'], inplace=True)
    
    # Add metadata columns
    labeled['train_period'] = train_period
    labeled['label_period'] = label_period
    
    gained = labeled['gained_fiber'].sum()
    total = len(labeled)
    print(f'  Gained fiber: {gained:,} / {total:,} ({100*gained/total:.2f}%)')
    
    training_sets.append(labeled)

# Concatenate all training pairs
training_data = pd.concat(training_sets, ignore_index=True)
print(f'\nTotal training data: {len(training_data):,} rows')

In [ ]:
# Class distribution
print('Class Distribution')
print('=' * 50)
positive = training_data['gained_fiber'].sum()
negative = len(training_data) - positive
total = len(training_data)

print(f'Positive (gained_fiber=1): {positive:,} ({100*positive/total:.2f}%)')
print(f'Negative (gained_fiber=0): {negative:,} ({100*negative/total:.2f}%)')
print(f'Imbalance ratio (neg/pos): {negative/max(positive,1):.1f}:1')
print(f'  → Recommended scale_pos_weight for XGBoost: {negative/max(positive,1):.1f}')

print(f'\nBreakdown by training pair:')
for (tp, lp), group in training_data.groupby(['train_period', 'label_period']):
    gained = group['gained_fiber'].sum()
    print(f'  {tp} → {lp}: {gained:,} / {len(group):,} ({100*gained/len(group):.2f}%)')

## Step 4: Handle Missing Values

In [ ]:
# Identify columns with NaN
print('Missing Values Analysis')
print('=' * 50)

null_summary = training_data.isnull().sum()
null_cols = null_summary[null_summary > 0]
if len(null_cols) > 0:
    print('Columns with missing values:')
    for col, count in null_cols.items():
        pct = 100 * count / len(training_data)
        print(f'  {col}: {count:,} ({pct:.2f}%)')
else:
    print('No missing values in training data!')

# Strategy:
# - Density features (housing_density, pop_density): NaN where area=0 → fill with 0
# - MHI: NaN where ACS data missing → leave as NaN (XGBoost handles natively)
# - Neighbor features: NaN for blocks with no neighbors → fill with 0

# Apply fills
fill_zero_cols = ['housing_density', 'pop_density', 'neighbor_fiber_pct', 'neighbor_count',
                  'projected_hh', 'projected_pop', 'hh_growth_rate']
for col in fill_zero_cols:
    if col in training_data.columns:
        before = training_data[col].isna().sum()
        training_data[col] = training_data[col].fillna(0)
        if before > 0:
            print(f'  Filled {col}: {before:,} NaN → 0')

# Also apply to full feature matrix
for col in fill_zero_cols:
    if col in feature_matrix.columns:
        feature_matrix[col] = feature_matrix[col].fillna(0)

# Report remaining NaN
remaining_nulls = training_data.isnull().sum()
remaining = remaining_nulls[remaining_nulls > 0]
if len(remaining) > 0:
    print(f'\nRemaining NaN (left for XGBoost to handle):')
    for col, count in remaining.items():
        print(f'  {col}: {count:,}')
else:
    print('\nAll NaN values resolved.')

## Step 5: Define Feature Column List

In [ ]:
# Metadata / ID columns (NOT features)
metadata_cols = [
    'filing_period', 'block_geoid', 'cbg_fips',
    'train_period', 'label_period',
    'has_fiber',  # this is the current status, not a predictive feature for no-fiber blocks
    'gained_fiber',  # target
]

# Feature columns = all columns minus metadata
feature_columns = [c for c in training_data.columns if c not in metadata_cols]

print(f'Feature columns ({len(feature_columns)}):')
for i, col in enumerate(feature_columns):
    dtype = training_data[col].dtype
    print(f'  {i+1:2d}. {col:40s} ({dtype})')

print(f'\nTarget: gained_fiber')
print(f'Metadata: {metadata_cols}')

In [ ]:
# Quick feature statistics
print('Feature Statistics (training data)')
print('=' * 60)
print(training_data[feature_columns].describe().T.to_string())

## Step 6: Export

In [ ]:
# 1. Full feature matrix (all periods, all CBs)
full_path = f'{OUTPUT_DIR}/feature_matrix_full.parquet'
feature_matrix.to_parquet(full_path, index=False)
print(f'Exported: {full_path}')
print(f'  Rows: {len(feature_matrix):,}  |  Columns: {len(feature_matrix.columns)}')
print(f'  Size: {os.path.getsize(full_path) / (1024*1024):.1f} MB')

# 2. Training data (labeled, no-fiber CBs only, 12-month pairs)
train_path = f'{OUTPUT_DIR}/training_data.parquet'
training_data.to_parquet(train_path, index=False)
print(f'\nExported: {train_path}')
print(f'  Rows: {len(training_data):,}  |  Columns: {len(training_data.columns)}')
print(f'  Size: {os.path.getsize(train_path) / (1024*1024):.1f} MB')

# 3. Feature columns list
cols_path = f'{OUTPUT_DIR}/feature_columns.json'
with open(cols_path, 'w') as f:
    json.dump(feature_columns, f, indent=2)
print(f'\nExported: {cols_path}')
print(f'  Features: {len(feature_columns)}')

print(f'\nPhase 4 complete!')

In [ ]:
# Final summary
print('=' * 60)
print('PHASE 4 SUMMARY')
print('=' * 60)
print(f'Full feature matrix: {len(feature_matrix):,} rows x {len(feature_matrix.columns)} cols')
print(f'Training data: {len(training_data):,} rows x {len(training_data.columns)} cols')
print(f'Feature columns: {len(feature_columns)}')
print(f'Target: gained_fiber')
print(f'Positive rate: {100*training_data["gained_fiber"].mean():.2f}%')
print(f'Imbalance ratio: {(1-training_data["gained_fiber"].mean())/max(training_data["gained_fiber"].mean(),0.001):.1f}:1')
print(f'\n12-month training pairs:')
for (tp, lp), group in training_data.groupby(['train_period', 'label_period']):
    print(f'  {tp} → {lp}: {len(group):,} samples, {group["gained_fiber"].sum():,} positive')
print(f'\nOutputs:')
print(f'  {full_path}')
print(f'  {train_path}')
print(f'  {cols_path}')